# Here we need to re-do the february experiment with 18hours as tp0 and use median metric instead

In [2]:
import polars as pl
from pathlib import Path
from subprocess import run
from circadian_cell_painting import (
    detect,
    CosinorDetection,
    CosinorVisualize,
    OscillationResults,
)

# analysis ready raw data
analysis_ready_dir = Path(
    "/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
    "Study/analysis_ready_data"
)

# datasets
data_dir = Path(
    "/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/preprocessed_data"
)

# platemaps
pm_path = Path(
    "/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
    "Study/platemaps/2026-03-16CellPaint_map.xlsx"
)
platemap = pl.read_excel(pm_path, engine="openpyxl")
# display(platemap)

analysis_results_dir = Path(
    "/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/results"
)

pickle_dir = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/pickles")

visual_results_dir = Path(
    "/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/visuals"
)

display(platemap)

WellName,stain set,cell line,condition
str,str,str,str
"""G12""","""Chroma""","""Bmal1""","""no treatment"""
"""G13""","""Chroma""","""Bmal1""","""no treatment"""
"""G14""","""Chroma""","""Bmal1""","""no treatment"""
"""G18""","""Chroma""","""Bmal1""","""0.1uM Dex"""
"""G19""","""Chroma""","""Bmal1""","""0.1uM Dex"""
…,…,…,…
"""J8""","""PhenoV""","""Per2""","""no treatment"""
"""J9""","""PhenoV""","""Per2""","""no treatment"""
"""K15""","""PhenoV""","""Per2""","""0.1uM Dex"""


preprocessing to fix the timepoints

In [8]:
timepoint_col = "Timepoint"
sampling_interval = 3
start_offset = 18
indexing = 1

for f in analysis_ready_dir.glob("2026-03*"):
    if "TIME-CORRECTED" in f.stem:
        continue
    
    curr_lf = (
        pl
        .scan_csv(f, separator="\t")
        .with_columns(
            (start_offset + (pl.col(timepoint_col) - indexing) * sampling_interval)
            .alias(timepoint_col).cast(pl.Float64)
            )
        )
    
    print(f.stem)
    
    out_file = analysis_ready_dir / f"{f.stem}._.TIME-CORRECTED.tsv"
    curr_lf.sink_csv(out_file, separator="\t")

2026-03-10CelPaint_ChromaNoTreatDex
2026-03-11CelPaint_PhenovNoTreatDex_HARMONY_FIXED


detrend + denoise again

In [15]:
window_hours = 24
n_timepoints = round(window_hours / sampling_interval)
print(n_timepoints)

well_col = "WellName"

for f in analysis_ready_dir.glob("2026-03*"):
    if 'TIME-CORRECTED' in f.stem:
        out_file = data_dir / f"{f.stem}.parquet"
        print(out_file)
        
        result = run([
            'ccellpaint', 'preprocess',
            str(f), str(out_file),
            '-w', well_col,
            '-t', timepoint_col,
            '--window', str(n_timepoints)
        ], capture_output=True, text=True)
        
        if result.returncode != 0:
            print(f"ERROR: {result.stderr}")
        else:
            print("✓ done")

8
/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/preprocessed_data/2026-03-10CelPaint_ChromaNoTreatDex._.TIME-CORRECTED.parquet
✓ done
/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/preprocessed_data/2026-03-11CelPaint_PhenovNoTreatDex_HARMONY_FIXED._.TIME-CORRECTED.parquet
✓ done


## Analysis portion

In [10]:
timepoint_col = "Timepoint"
sampling_interval = 3
start_offset = 18
indexing = 1
well_col = "WellName"
stat_val = "median"
p_thresh = 0.05

model = CosinorDetection()

for f in data_dir.glob("2026-03*"):
    if "TIME-CORRECTED" in f.stem:
        file_name = f.stem
        
        out_file = analysis_results_dir / f"{file_name}._.RESULTS.parquet"
        
        result: OscillationResults = detect(
            file_path=f,
            platemap_path=platemap,
            out_file=out_file,
            well_column=well_col,
            feature_col="feature",
            timepoint_col=timepoint_col,
            value_col=stat_val,
            stat=stat_val,
            correction_group=["cell line", "condition"],
            detector=model,
        )
        
        pickle_out = pickle_dir / f"{file_name}._.RESULT_OBJ.pickle"
        result.to_pickle(pickle_out)
        
        # convert and filter results
        csv_out = analysis_results_dir / f"{file_name}._.P-filtered.csv"
        (pl
         .scan_parquet(out_file)
         .filter(pl.col("p_adjusted") <= p_thresh)
         .drop("fitted")
         .sink_csv(csv_out)
         )

Well progress:   0%|          | 0/12 [00:00<?, ?it/s]

Well progress: 100%|██████████| 12/12 [01:56<00:00,  9.69s/it]


## Visualization portion

In [ ]:
for pickles, dataset in zip(pickle_dir.glob("2026-03*"), data_dir.glob("2026-03*")):
    if "TIME-CORRECTED" in pickles.stem:
        results_obj = OscillationResults.load_pickle(pickles)
        
        visualize_out = visual_results_dir / dataset.stem
        print(visualize_out)
        
        visualize = CosinorVisualize(results_obj, tick_interval=6)
        visualize.view(visualize_out)

/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/visuals/2026-03-10CelPaint_ChromaNoTreatDex._.TIME-CORRECTED


well_processing: 100%|██████████| 8/8 [00:16<00:00,  2.08s/it]


/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/visuals/2026-03-11CelPaint_PhenovNoTreatDex_HARMONY_FIXED._.TIME-CORRECTED


well_processing:  56%|█████▌    | 5/9 [01:38<01:18, 19.63s/it]


KeyboardInterrupt: 